<a href="https://colab.research.google.com/github/Asm2910/multimodal_rag/blob/main/Multimodal_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
ROOT = "/content/drive/MyDrive/rag_data"
m = json.load(open(f"{ROOT}/manifest.json"))
print(len(m), "records")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
4777 records


In [ ]:
!ls {ROOT}
!ls {ROOT}/images | head -3

images	manifest.json  pdfs
2311.08493_p2_img0.png
2311.08493_p4_img0.png
2311.08493_p4_img1.png


In [ ]:
from PIL import Image
from collections import defaultdict

MAX_PER_PAPER = 10
MIN_PIXELS = 300 * 300

by_doc = defaultdict(list)
for r in m:
    if r["type"] == "text":
        continue
    with Image.open(r["path"]) as im:
        w, h = im.size
    if r["type"] == "image" and w * h < MIN_PIXELS:
        continue
    r["pixels"] = w * h
    by_doc[r["doc_id"]].append(r)

selected = []
for doc, recs in by_doc.items():
    charts = [x for x in recs if x["type"] == "chart_page"]
    images = sorted((x for x in recs if x["type"] == "image"),
                    key=lambda x: -x["pixels"])
    selected += charts + images[:max(0, MAX_PER_PAPER - len(charts))]

json.dump(selected, open(f"{ROOT}/selected_images.json", "w"), indent=2)
print("vision calls needed:", len(selected))   # should print 717

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (114843746 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


vision calls needed: 717


In [ ]:
!pip install -q google-genai pinecone sentence-transformers langchain-text-splitters pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 26.4 MB/s eta 0:00:00


In [ ]:
%%writefile ingest.py

"""
Multimodal RAG over research papers: Gemini (vision) + local embeddings + Pinecone,
with hybrid retrieval (BM25 keyword search + vector search, fused by rank).

Setup (Colab):
    !pip install -q google-genai pinecone sentence-transformers \
        langchain-text-splitters pillow rank-bm25

Keys: set GEMINI_API_KEY and PINECONE_API_KEY in os.environ.
Reads from /content/drive/MyDrive/rag_data by default.

Stages (run in order):
    python ingest.py describe --limit 3    # tiny test first
    python ingest.py describe              # all selected images, resumable
    python ingest.py index                 # chunk + embed + rebuild Pinecone namespace
    python ingest.py ask "your question" [--k 6] [--no-answer]
"""
import argparse
import json
import os
import pathlib
import re
import time
from collections import defaultdict

ROOT = pathlib.Path(os.environ.get("RAG_ROOT", "/content/drive/MyDrive/rag_data"))
VISION_MODEL = os.environ.get("VISION_MODEL", "gemini-3.1-flash-lite-preview")
ANSWER_MODEL = os.environ.get("ANSWER_MODEL", VISION_MODEL)
EMBED_MODEL = "BAAI/bge-small-en-v1.5"   # 384 dims, free, runs on CPU
EMBED_DIM = 384
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
INDEX_NAME = "multimodal-rag"
NAMESPACE = "papers"
DELAY = float(os.environ.get("REQUEST_DELAY", "4"))
SUMMARY_FILE = ROOT / "summaries.jsonl"
CHUNK_FILE = ROOT / "chunks.jsonl"       # local copy of every chunk, used for BM25
CHUNK_SIZE = 600
CHUNK_OVERLAP = 100
CANDIDATES = 40                          # fetched from each retriever before fusion

PROMPT = (
    "You are describing a figure from a research paper so it can be found by "
    "text search. In under 120 words: state the figure type (line chart, bar "
    "chart, table, architecture diagram, photo/medical image, etc.), what it "
    "shows, axis labels and units, the main trend or comparison, and any key "
    "numbers or model names visible. If the image is a full page, describe only "
    "the figures and tables on it, not the body text. Do not guess values you "
    "cannot read."
)


# ---------------------------------------------------------------- describe
def load_summaries() -> dict:
    done = {}
    if SUMMARY_FILE.exists():
        with SUMMARY_FILE.open() as f:
            for line in f:
                line = line.strip()
                if line:
                    r = json.loads(line)
                    done[r["path"]] = r
    return done


def describe(limit: int | None = None) -> None:
    from google import genai
    from google.genai import errors
    from PIL import Image

    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    selected = json.load(open(ROOT / "selected_images.json"))
    done = load_summaries()
    todo = [r for r in selected if r["path"] not in done]
    if limit:
        todo = todo[:limit]
    print(f"{len(done)} already described, {len(todo)} to do "
          f"(model: {VISION_MODEL})")

    skipped = 0
    with SUMMARY_FILE.open("a") as out:
        for n, rec in enumerate(todo, 1):
            summary, quota_exhausted = None, False
            for attempt in range(3):
                try:
                    with Image.open(rec["path"]) as im:
                        im = im.convert("RGB")
                        resp = client.models.generate_content(
                            model=VISION_MODEL, contents=[im, PROMPT]
                        )
                    summary = (resp.text or "").strip()
                    break
                except errors.APIError as e:
                    if e.code == 429:
                        print(f"  rate limited (attempt {attempt + 1}/3), "
                              "waiting 65s")
                        time.sleep(65)
                        quota_exhausted = attempt == 2
                    elif e.code in (500, 503):
                        time.sleep(10)
                    else:
                        print(f"  API error {e.code}: {e.message}")
                        break
            if quota_exhausted:
                print("Quota looks exhausted. Progress is saved; rerun later "
                      "and it will resume.")
                return
            if not summary:
                skipped += 1
                continue
            out.write(json.dumps({
                "path": rec["path"], "doc_id": rec["doc_id"],
                "page": rec["page"], "type": rec["type"], "summary": summary,
            }) + "\n")
            out.flush()
            if n <= 3 or n % 25 == 0:
                print(f"[{n}/{len(todo)}] {rec['doc_id']} p{rec['page']}: "
                      f"{summary[:120]}...")
            time.sleep(DELAY)
    print(f"done. skipped (empty/blocked): {skipped}")


# ------------------------------------------------------------------- index
def get_index():
    from pinecone import Pinecone, ServerlessSpec

    pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
    if INDEX_NAME not in pc.list_indexes().names():
        pc.create_index(
            name=INDEX_NAME, dimension=EMBED_DIM, metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
        while not pc.describe_index(INDEX_NAME).status["ready"]:
            time.sleep(2)
    return pc.Index(INDEX_NAME)


def index() -> None:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from sentence_transformers import SentenceTransformer

    manifest = json.load(open(ROOT / "manifest.json"))
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
    )

    items = []
    for r in manifest:
        if r["type"] != "text":
            continue
        for i, chunk in enumerate(splitter.split_text(r["content"])):
            if len(chunk.strip()) < 60:
                continue
            items.append({
                "id": f'{r["doc_id"]}-p{r["page"]}-t{i}',
                "text": chunk,
                "meta": {"doc_id": r["doc_id"], "page": r["page"],
                         "type": "text", "image_path": "", "text": chunk},
            })
    n_text = len(items)

    for r in load_summaries().values():
        stem = pathlib.Path(r["path"]).stem
        items.append({
            "id": f'{r["doc_id"]}-p{r["page"]}-{stem}',
            "text": r["summary"],
            "meta": {"doc_id": r["doc_id"], "page": r["page"],
                     "type": r["type"], "image_path": r["path"],
                     "text": r["summary"]},
        })
    print(f"{n_text} text chunks + {len(items) - n_text} image summaries")

    # Local copy of all chunks, so keyword search can run without Pinecone.
    with CHUNK_FILE.open("w") as f:
        for it in items:
            f.write(json.dumps({"id": it["id"], "meta": it["meta"]}) + "\n")

    model = SentenceTransformer(EMBED_MODEL)
    vectors = model.encode([it["text"] for it in items], batch_size=64,
                           normalize_embeddings=True, show_progress_bar=True)

    idx = get_index()
    try:  # clean rebuild so stale chunks from an older chunk size are removed
        idx.delete(delete_all=True, namespace=NAMESPACE)
    except Exception as e:
        print("(nothing to delete yet)", type(e).__name__)

    for start in range(0, len(items), 100):
        batch = [
            {"id": it["id"], "values": vec.tolist(), "metadata": it["meta"]}
            for it, vec in zip(items[start:start + 100], vectors[start:start + 100])
        ]
        idx.upsert(vectors=batch, namespace=NAMESPACE)
    print("upserted", len(items), "vectors")
    print(idx.describe_index_stats())


# ----------------------------------------------------------- hybrid search
_TOKEN = re.compile(r"[a-z0-9]+(?:[.\-][a-z0-9]+)*")


def tokenize(text: str) -> list[str]:
    # keeps tokens like "76.39", "bi-rads", "convnext" intact
    return _TOKEN.findall(text.lower())


_bm25_cache = {}


def bm25_search(question: str, n: int):
    from rank_bm25 import BM25Okapi

    if "bm25" not in _bm25_cache:
        rows = [json.loads(l) for l in CHUNK_FILE.open() if l.strip()]
        _bm25_cache["rows"] = rows
        _bm25_cache["bm25"] = BM25Okapi([tokenize(r["meta"]["text"]) for r in rows])
    rows, bm25 = _bm25_cache["rows"], _bm25_cache["bm25"]
    scores = bm25.get_scores(tokenize(question))
    order = sorted(range(len(rows)), key=lambda i: -scores[i])[:n]
    return [(rows[i]["id"], rows[i]["meta"]) for i in order if scores[i] > 0]


def rrf(rank_lists: list[list[str]], k: int = 60) -> dict:
    """Reciprocal rank fusion: score = sum of 1/(k + rank) across lists."""
    scores = defaultdict(float)
    for lst in rank_lists:
        for rank, _id in enumerate(lst):
            scores[_id] += 1.0 / (k + rank + 1)
    return scores


_cache = {}


def get_embedder():
    if "emb" not in _cache:
        from sentence_transformers import SentenceTransformer
        _cache["emb"] = SentenceTransformer(EMBED_MODEL)
    return _cache["emb"]


def get_cached_index():
    if "idx" not in _cache:
        _cache["idx"] = get_index()
    return _cache["idx"]


def retrieve(question: str, k: int = 6, mode: str = "hybrid"):
    """mode: 'hybrid' (default), 'vector', or 'keyword'.
    Returns [(id, metadata, fused_score, vector_rank, keyword_rank), ...]."""
    vec_hits, kw_hits = [], []
    if mode in ("hybrid", "vector"):
        qvec = get_embedder().encode(
            QUERY_PREFIX + question, normalize_embeddings=True
        ).tolist()
        res = get_cached_index().query(vector=qvec, top_k=CANDIDATES,
                                       namespace=NAMESPACE, include_metadata=True)
        vec_hits = [(m.id, dict(m.metadata)) for m in res.matches]
    if mode in ("hybrid", "keyword") and CHUNK_FILE.exists():
        kw_hits = bm25_search(question, CANDIDATES)

    meta = {i: md for i, md in vec_hits}
    meta.update({i: md for i, md in kw_hits})
    vec_rank = {i: r + 1 for r, (i, _) in enumerate(vec_hits)}
    kw_rank = {i: r + 1 for r, (i, _) in enumerate(kw_hits)}

    fused = rrf([[i for i, _ in vec_hits], [i for i, _ in kw_hits]])
    top = sorted(fused, key=lambda i: -fused[i])[:k]
    return [(i, meta[i], fused[i], vec_rank.get(i), kw_rank.get(i)) for i in top]


# ------------------------------------------------------------------ answer
def answer_question(question: str, k: int = 6, mode: str = "hybrid"):
    """Retrieve, then ask Gemini. Returns (answer_text, hits)."""
    from google import genai
    from PIL import Image

    hits = retrieve(question, k, mode)
    context, images = [], []
    for n, (_id, md, *_rest) in enumerate(hits, 1):
        context.append(f"[{n}] ({md['type']}, paper {md['doc_id']}, page "
                       f"{md['page']}): {md['text']}")
        if md["image_path"] and len(images) < 3 and os.path.exists(md["image_path"]):
            images.append(Image.open(md["image_path"]).convert("RGB"))

    prompt = (
        "Answer the question using only the context below (and any attached "
        "figures). Cite sources only as bracketed numbers like [1] or [2][5], "
        "matching the numbered context items; never write 'Page 8' or similar "
        "as a citation. If the context is not enough, say so.\n\n"
        "CONTEXT:\n" + "\n\n".join(context) + f"\n\nQUESTION: {question}"
    )
    if "gemini" not in _cache:
        _cache["gemini"] = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    resp = _cache["gemini"].models.generate_content(
        model=ANSWER_MODEL, contents=[prompt] + images
    )
    return (resp.text or "(empty response)"), hits


# --------------------------------------------------------------------- ask
def ask(question: str, k: int = 6, answer: bool = True) -> None:
    if answer:
        text, hits = answer_question(question, k)
    else:
        text, hits = None, retrieve(question, k)

    print("TOP MATCHES (vec = vector rank, kw = keyword rank)")
    for _id, md, score, vr, kr in hits:
        print(f"  {score:.4f}  {md['type']:<10} {md['doc_id']} p{md['page']}  "
              f"vec={vr or '-'} kw={kr or '-'}")
        print(f"         {md['text'][:140].replace(chr(10), ' ')}...")
    if text is not None:
        print("\nANSWER\n" + text)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    d = sub.add_parser("describe")
    d.add_argument("--limit", type=int, default=None)
    sub.add_parser("index")
    a = sub.add_parser("ask")
    a.add_argument("question")
    a.add_argument("--k", type=int, default=6)
    a.add_argument("--no-answer", action="store_true")
    args = ap.parse_args()

    if args.cmd == "describe":
        describe(args.limit)
    elif args.cmd == "index":
        index()
    else:
        ask(args.question, k=args.k, answer=not args.no_answer)

Overwriting ingest.py


In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!python ingest.py describe --limit 3

0 already described, 3 to do (model: gemini-3.5-flash-lite)
[1/3] 2311.08493 p2: This bar chart illustrates the image distribution across BI-RADS categories. 

The x-axis displays the "Category" with v...
[2/3] 2311.08493 p4: This figure is a medical ultrasound image showing a hypoechoic, well-circumscribed breast lesion with a lobulated shape ...
[3/3] 2311.08493 p4: This is a **medical ultrasound image** showing a breast lesion. 

The image displays a well-circumscribed, oval to lobul...
done. skipped (empty/blocked): 0


In [ ]:
!python ingest.py describe --limit 150

499 already described, 218 to do (model: gemini-3.5-flash-lite)
  rate limited (attempt 1/3), waiting 65s
  rate limited (attempt 2/3), waiting 65s
  rate limited (attempt 3/3), waiting 65s
Quota looks exhausted. Progress is saved; rerun later (e.g. tomorrow) and it will resume.


In [ ]:
import os
os.environ["VISION_MODEL"] = "gemini-3.1-flash-lite-preview"
!python ingest.py describe

499 already described, 218 to do (model: gemini-3.1-flash-lite-preview)
[1/218] 2409.13568 p20: This figure displays a confusion matrix and a corresponding horizontal bar chart ("Class Representation") for the PASTIS...
[2/218] 2609.20100 p2: This figure is a schematic illustration showing three landscape plots representing the optimization or representational ...
[3/218] 2609.20100 p3: This image displays three heatmaps representing the pairwise cosine similarity between block outputs of Transformer mode...
[25/218] 2609.21012 p14: This figure displays a series of heatmaps visualizing Shapley/Owen values, which represent model attribution for an artw...
[50/218] 2609.22105 p4: This figure is a stylized conceptual diagram illustrating signal processing or filtering. It features a complex, high-fr...
[75/218] 2609.22604 p18: This figure contains six false-color microscopy-based phase maps (labeled a–f) representing the microstructural evolutio...
[100/218] 2609.23397 p5: This is an arch

In [ ]:
!pip install -q rank-bm25

In [ ]:
!python ingest.py index

13099 text chunks + 716 image summaries
Loading weights: 100% 199/199 [00:00<00:00, 8240.61it/s]
Batches: 100% 216/216 [01:15<00:00,  2.88it/s]
upserted 13815 vectors
DescribeIndexStatsResponse(dimension=384, total_vector_count=13815, metric='cosine', namespaces=1)


In [ ]:
# !python ingest.py ask "Which figure compares feature importance across models on the Pima Indians Diabetes dataset?"
# !python ingest.py ask "What did the crop scale ablation show for DINOv1 loss?"
!python ingest.py ask "What accuracy did ConvNeXt achieve on BI-RADS classification?"

Loading weights: 100% 199/199 [00:00<00:00, 5450.31it/s]
TOP MATCHES (vec = vector rank, kw = keyword rank)
  0.0323  text       2311.08493 p1  vec=1 kw=3
         process. This research endeavour aims to bridge this gap, offering a comprehensive evaluation of the potential synergies between machine lea...
  0.0320  text       2506.08641 p27  vec=4 kw=1
         Unlocking Pretrained Vision Transformers for Time Series Classification 27 and thus the only setting we can identify with a convolutional ba...
  0.0320  text       2311.08493 p1  vec=3 kw=2
         ear evaluation, and training from scratch. Full fine-tuning achieved the strongest overall performance, with ConvNeXt attaining an accuracy ...
  0.0308  text       2311.08493 p3  vec=2 kw=8
         > linear evaluation > training from scratch. ConvNeXt achieved the highest accuracy and F1 score under all three training strategies. Under ...
  0.0303  text       2311.08493 p1  vec=8 kw=4
         Performance of Machine Learning Cla

In [ ]:
!python ingest.py ask "Which figure compares feature importance across models on the Pima Indians Diabetes dataset?"

!python ingest.py ask "What did the crop scale ablation show for DINOv1 loss?"

Loading weights: 100% 199/199 [00:00<00:00, 12919.73it/s]
TOP MATCHES (vec = vector rank, kw = keyword rank)
  0.0328  image      2609.29650 p9  vec=1 kw=1
         This bar chart compares feature importance rankings for the "Pima Indians Diabetes" dataset across five models: VG-Degree, Random Forest, XG...
  0.0323  text       2609.29650 p5  vec=2 kw=2
         Grad-CAM Figure 3 shows the feature importance rankings produced by five methods for PIMA and WDBC, where each bar represents the rank of a ...
  0.0313  text       2609.29650 p4  vec=5 kw=3
         attended to equally regardless of class and are therefore less discriminative. 4 RESULTS In this section, we present the datasets used, expe...
  0.0303  image      2609.29650 p9  vec=6 kw=6
         This figure is a grouped bar chart titled "Feature ranking comparison — BC Wisconsin Diagnostic." It compares the importance of 30 different...
  0.0301  text       2609.29650 p6  vec=8 kw=5
         0.8138 ± 0.0335 0.9277 ± 0.0453 0.9

In [ ]:
import json
m = json.load(open("/content/drive/MyDrive/rag_data/manifest.json"))
hits = [r for r in m if r["type"] == "text" and "76.39" in r["content"]]
print([(r["doc_id"], r["page"]) for r in hits])

[('2311.08493', 1), ('2311.08493', 3), ('2311.08493', 4), ('2609.12825', 6)]


In [ ]:
import sys; sys.path.append("/content")
import ingest
ingest.ask("What accuracy did ConvNeXt achieve on BI-RADS classification?", k=15)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

TOP MATCHES
  0.760  text       2311.08493 p1
         proven malignancy. Category 4 is further subdivided into 4A, 4B, and 4C, reflecting increasing levels of suspicion for malignancy. This, in ...
  0.750  text       2311.08493 p1
         Performance of Machine Learning Classification in Sonomammogram Images Using BI-RADS Malitha Gunawardhana Auckland Bioengineering Institute ...
  0.750  text       2506.08641 p27
         Unlocking Pretrained Vision Transformers for Time Series Classification 27 and thus the only setting we can identify with a convolutional ba...
  0.747  text       2603.29660 p20
         (73% with logistic regression). This contrasts different rep- resentations and classifiers, not a statistically established advantage. 3http...
  0.735  text       2609.11692 p27
         the minimum validation loss does not provide a direct measure of the expected test set ranking across architectures. In particular, the clas...
  0.733  text       2604.05819 p25
         +1.9% 

In [ ]:
# import os
# from google import genai
# from google.genai import errors

# client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# available = [m.name.replace("models/", "") for m in client.models.list()]
# candidates = [n for n in available
#               if "flash" in n and "image" not in n and "tts" not in n
#               and "live" not in n and "audio" not in n]
# print("flash models on your key:", candidates, "\n")

# working = []
# for name in candidates:
#     try:
#         client.models.generate_content(model=name, contents="Say OK")
#         print("works  ", name)
#         working.append(name)
#     except errors.APIError as e:
#         print(f"failed ({e.code})", name)

In [ ]:
%%writefile eval.py

"""
Evaluate retrieval quality on a small question set.

Usage (Colab, after keys are in os.environ):
    !python eval.py                      # compares vector / keyword / hybrid, free
    !python eval.py --k 10               # use top-10 instead of top-6
    !python eval.py --answers            # also generate answers (uses Gemini quota)

Question file format (eval_questions.json), each item:
    {
      "question": "...",
      "doc_id": "2311.08493",          # optional: expected paper
      "page": [8],                      # optional: acceptable page number(s)
      "needs": "figure" | "text",       # optional: expected source kind
      "keyword": "76.39",               # optional: must appear in a retrieved chunk
      "answer_contains": "76.39"        # optional: used only with --answers
    }
Only the fields you provide are scored. Retrieval-only runs make no LLM calls.
"""
import argparse
import json
import os
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parent))
import ingest  # noqa: E402

HERE = pathlib.Path(__file__).resolve().parent
FIGURE_TYPES = {"image", "chart_page"}


def score_question(q: dict, hits: list) -> dict:
    mds = [md for _id, md, *_ in hits]
    res = {}
    if "doc_id" in q:
        res["doc"] = any(md["doc_id"] == q["doc_id"] for md in mds)
    if "page" in q:
        pages = q["page"] if isinstance(q["page"], list) else [q["page"]]
        res["page"] = any(
            md["doc_id"] == q.get("doc_id", md["doc_id"]) and md["page"] in pages
            for md in mds
        )
    if "needs" in q:
        if q["needs"] == "figure":
            res["kind"] = any(md["type"] in FIGURE_TYPES for md in mds)
        else:
            res["kind"] = any(md["type"] == "text" for md in mds)
    if "keyword" in q:
        kw = q["keyword"].lower()
        res["keyword"] = any(kw in md["text"].lower() for md in mds)
    return res


def run_mode(questions: list, mode: str, k: int, answers: bool) -> list:
    rows = []
    for q in questions:
        hits = ingest.retrieve(q["question"], k=k, mode=mode)
        scores = score_question(q, hits)
        if answers and "answer_contains" in q:
            text, _ = ingest.answer_question(q["question"], k=k, mode=mode)
            scores["answer"] = q["answer_contains"].lower() in text.lower()
        rows.append((q["question"], scores))
    return rows


def summarize(rows: list) -> dict:
    totals = {}
    for _q, scores in rows:
        for metric, ok in scores.items():
            hit, n = totals.get(metric, (0, 0))
            totals[metric] = (hit + int(ok), n + 1)
    return totals


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--file", default=str(HERE / "eval_questions.json"))
    ap.add_argument("--k", type=int, default=6)
    ap.add_argument("--answers", action="store_true",
                    help="also generate answers and check answer_contains")
    args = ap.parse_args()

    questions = json.load(open(args.file))
    print(f"{len(questions)} questions, top-{args.k}\n")

    summary = {}
    details = {}
    for mode in ("vector", "keyword", "hybrid"):
        rows = run_mode(questions, mode, args.k, args.answers and mode == "hybrid")
        summary[mode] = summarize(rows)
        details[mode] = rows

    metrics = ["doc", "page", "kind", "keyword", "answer"]
    print(f"{'mode':<9}" + "".join(f"{m:>10}" for m in metrics))
    for mode, totals in summary.items():
        line = f"{mode:<9}"
        for m in metrics:
            if m in totals:
                hit, n = totals[m]
                line += f"{f'{hit}/{n}':>10}"
            else:
                line += f"{'-':>10}"
        print(line)

    print("\nMisses in hybrid mode:")
    any_miss = False
    for question, scores in details["hybrid"]:
        failed = [m for m, ok in scores.items() if not ok]
        if failed:
            any_miss = True
            print(f"  - {question}\n    failed: {', '.join(failed)}")
    if not any_miss:
        print("  none")


if __name__ == "__main__":
    main()

Writing eval.py


In [ ]:
%%writefile eval_questions.json


[
  {
    "question": "What accuracy did ConvNeXt achieve on BI-RADS classification?",
    "doc_id": "2311.08493",
    "needs": "text",
    "keyword": "76.39",
    "answer_contains": "76.39"
  },
  {
    "question": "How many images were used for training and validation in the BI-RADS sonomammogram study?",
    "doc_id": "2311.08493",
    "needs": "text",
    "keyword": "2,945",
    "answer_contains": "2,945"
  },
  {
    "question": "How many women were diagnosed with breast cancer in 2020 according to the WHO?",
    "doc_id": "2311.08493",
    "needs": "text",
    "keyword": "2.3 million",
    "answer_contains": "2.3 million"
  },
  {
    "question": "Which figure compares feature importance across models on the Pima Indians Diabetes dataset?",
    "doc_id": "2609.29650",
    "page": [9],
    "needs": "figure"
  },
  {
    "question": "Which figure shows the feature ranking comparison for the Wisconsin Diagnostic breast cancer dataset?",
    "doc_id": "2609.29650",
    "page": [9],
    "needs": "figure"
  },
  {
    "question": "What did the crop scale ablation show for DINOv1 loss?",
    "doc_id": "2609.28187",
    "page": [8],
    "needs": "figure"
  },
  {
    "question": "Where are the complete metrics for the fixed-scale crop ablation?",
    "doc_id": "2609.28187",
    "page": [17],
    "needs": "text"
  },
  {
    "question": "Which five methods are compared in the feature importance rankings?",
    "doc_id": "2609.29650",
    "page": [5, 9],
    "needs": "text",
    "keyword": "visibility graph"
  }
]

Writing eval_questions.json


In [ ]:
import json
print(len(json.load(open("/content/eval_questions.json"))), "questions")

8 questions


In [ ]:
%%writefile App.py

"""
Gradio demo for the multimodal paper RAG.

Usage (Colab, after keys are in os.environ and the index is built):
    !pip install -q gradio
    !python app.py

The cell keeps running and prints a public link (https://....gradio.live).
Stop the cell to shut the demo down.
"""
import os
import pathlib
import sys

import gradio as gr

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parent))
import ingest  # noqa: E402

EXAMPLES = [
    ["What accuracy did ConvNeXt achieve on BI-RADS classification?"],
    ["Which figure compares feature importance across models on the Pima Indians Diabetes dataset?"],
    ["What did the crop scale ablation show for DINOv1 loss?"],
]


def run(question: str, k: int, mode: str):
    if not question or not question.strip():
        return "Please enter a question.", [], []
    try:
        answer, hits = ingest.answer_question(question, k=int(k), mode=mode)
    except Exception as e:  # show quota / key problems in the UI, not a stack trace
        return f"Error: {type(e).__name__}: {e}", [], []

    gallery = [
        (md["image_path"], f'{md["doc_id"]} p{md["page"]} ({md["type"]})')
        for _id, md, *_ in hits
        if md["image_path"] and os.path.exists(md["image_path"])
    ]
    rows = [
        [n, md["type"], md["doc_id"], md["page"],
         vr if vr else "", kr if kr else "",
         md["text"][:220].replace("\n", " ")]
        for n, (_id, md, _score, vr, kr) in enumerate(hits, 1)
    ]
    return answer, gallery, rows


def build_demo() -> gr.Blocks:
    with gr.Blocks(title="Multimodal paper RAG") as demo:
        gr.Markdown(
            "# Multimodal RAG over research papers\n"
            "Ask about text, tables, or figures. Figures are searched through "
            "AI-written descriptions, and the original images are shown below."
        )
        with gr.Row():
            question = gr.Textbox(label="Question", lines=2, scale=4)
            with gr.Column(scale=1):
                k = gr.Slider(3, 12, value=6, step=1, label="Top-k")
                mode = gr.Radio(["hybrid", "vector", "keyword"], value="hybrid",
                                label="Retrieval mode")
        btn = gr.Button("Ask", variant="primary")
        answer = gr.Markdown(label="Answer")
        gallery = gr.Gallery(label="Retrieved figures", columns=3, height="auto")
        table = gr.Dataframe(
            headers=["#", "type", "paper", "page", "vec rank", "kw rank", "text"],
            label="Retrieved passages (the # matches the [n] citations)",
            wrap=True,
        )
        gr.Examples(EXAMPLES, inputs=[question])
        btn.click(run, [question, k, mode], [answer, gallery, table])
        question.submit(run, [question, k, mode], [answer, gallery, table])
    return demo


if __name__ == "__main__":
    build_demo().launch(share=True)

Writing App.py


In [ ]:
%%writefile lc_retriever.py

"""
LangChain wrapper around the hybrid (BM25 + vector) retriever.

Setup:
    !pip install -q langchain-core
    (optional chain example also needs: !pip install -q langchain-google-genai)

Usage:
    import lc_retriever
    retriever = lc_retriever.HybridRetriever(k=6)
    docs = retriever.invoke("What accuracy did ConvNeXt achieve on BI-RADS?")
    for d in docs:
        print(d.metadata["doc_id"], d.metadata["page"], d.metadata["type"])
        print(d.page_content[:150])

Figure results are returned as Documents whose page_content is the AI-written
description and whose metadata["image_path"] points to the original image.
"""
import pathlib
import sys

from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parent))
import ingest  # noqa: E402


class HybridRetriever(BaseRetriever):
    """Hybrid keyword + vector retriever over the Pinecone index."""

    k: int = 6
    mode: str = "hybrid"  # "hybrid" | "vector" | "keyword"

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> list[Document]:
        hits = ingest.retrieve(query, k=self.k, mode=self.mode)
        docs = []
        for _id, md, score, vec_rank, kw_rank in hits:
            meta = {key: val for key, val in md.items() if key != "text"}
            meta.update({"id": _id, "fused_score": score,
                         "vec_rank": vec_rank, "kw_rank": kw_rank})
            docs.append(Document(page_content=md["text"], metadata=meta))
        return docs


def build_chain(model: str | None = None):
    """Optional text-only LCEL chain: retriever -> prompt -> Gemini -> text.
    (Image attachments are handled by ingest.answer_question / app.py.)
    Requires langchain-google-genai and GEMINI_API_KEY (or GOOGLE_API_KEY)."""
    import os

    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnableLambda, RunnablePassthrough
    from langchain_google_genai import ChatGoogleGenerativeAI

    llm = ChatGoogleGenerativeAI(
        model=model or ingest.ANSWER_MODEL,
        google_api_key=os.environ.get("GEMINI_API_KEY"),
    )
    prompt = ChatPromptTemplate.from_template(
        "Answer using only the context. Cite sources only as bracketed numbers "
        "like [1] matching the numbered items. If the context is not enough, "
        "say so.\n\nCONTEXT:\n{context}\n\nQUESTION: {question}"
    )

    def format_docs(docs: list[Document]) -> str:
        return "\n\n".join(
            f"[{n}] ({d.metadata['type']}, paper {d.metadata['doc_id']}, "
            f"page {d.metadata['page']}): {d.page_content}"
            for n, d in enumerate(docs, 1)
        )

    retriever = HybridRetriever()
    return (
        {"context": retriever | RunnableLambda(format_docs),
         "question": RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )

Writing lc_retriever.py


In [ ]:
!python eval.py

8 questions, top-6

Loading weights: 100% 199/199 [00:00<00:00, 5456.58it/s]
mode            doc      page      kind   keyword    answer
vector          8/8       5/5       8/8       4/4         -
keyword         8/8       5/5       8/8       4/4         -
hybrid          8/8       5/5       8/8       4/4         -

Misses in hybrid mode:
  none


In [ ]:
# !pip install -q gradio
# !python App.py

In [ ]:
!python eval.py --k 3

8 questions, top-3

Loading weights: 100% 199/199 [00:00<00:00, 5379.91it/s]
mode            doc      page      kind   keyword    answer
vector          8/8       5/5       8/8       4/4         -
keyword         8/8       4/5       8/8       4/4         -
hybrid          8/8       5/5       8/8       4/4         -

Misses in hybrid mode:
  none


In [ ]:
!python eval.py --answers

8 questions, top-6

Loading weights: 100% 199/199 [00:00<00:00, 5214.48it/s]
mode            doc      page      kind   keyword    answer
vector          8/8       5/5       8/8       4/4         -
keyword         8/8       5/5       8/8       4/4         -
hybrid          8/8       5/5       8/8       4/4       3/3

Misses in hybrid mode:
  none
